# 1. SPARK MACHINE LEARNING & PIPELINES
- Môn học: Big Data - CO3137
- Ngày 20/05/2026
- Lớp: L01

| STT | Họ tên | MSSV |
| :---: | :--- | :---: |
| 1 | Lê Đình Đức | 2310774 |
| 2 | Nguyễn Văn Công Thành | 231xxx |

# 3. Exercise

### Exercise 0: Prepare movie data

In [ ]:
import kagglehub
from confluent_kafka.admin import AdminClient, NewTopic
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

KAFKA_BROKERS = "localhost:9092,localhost:9192,localhost:9292"

spark = (SparkSession.builder.appName("Lab4")
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.1.1,org.apache.kafka:kafka-clients:3.6.0,org.apache.spark:spark-streaming-kafka-0-10_2.13:4.1.1")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "8")
    .master("local[*]")
    .getOrCreate())

spark.sparkContext.setLogLevel("ERROR")
print("Spark Session initialized successfully.")


In [ ]:
# Download MovieLens latest-small dataset
path = kagglehub.dataset_download("grouplens/movielens-latest-small")

df_ratings_csv = spark.read.csv(path + "/ratings.csv", header=True, inferSchema=True)
df_movies_csv = spark.read.csv(path + "/movies.csv", header=True, inferSchema=True)
df_tags_csv = spark.read.csv(path + "/tags.csv", header=True, inferSchema=True)

print("Ratings preview:")
df_ratings_csv.show(3)
print("Movies preview:")
df_movies_csv.show(3)
print("Tags preview:")
df_tags_csv.show(3)

# Delete existing topics and create them anew on local Kafka brokers
admin_client = AdminClient({'bootstrap.servers': KAFKA_BROKERS})
try:
    admin_client.delete_topics(['Lab1_ratings', 'Lab1_movies', 'Lab1_tags'], operation_timeout=10)
    print("Deleted existing topics (if any)")
except Exception as e:
    print(f"Notice: delete_topics encountered: {e}")

for rep_factor in [2, 1]:
    new_topics = [
        NewTopic(topic="Lab1_ratings", num_partitions=3, replication_factor=rep_factor),
        NewTopic(topic="Lab1_movies", num_partitions=3, replication_factor=rep_factor),
        NewTopic(topic="Lab1_tags", num_partitions=3, replication_factor=rep_factor)
    ]
    fs = admin_client.create_topics(new_topics)
    success = True
    for topic, f in fs.items():
        try:
            f.result() # Wait for topic generation
            print(f"Topic '{topic}' created successfully with replication factor {rep_factor}.")
        except Exception as e:
            if "INVALID_REPLICATION_FACTOR" in str(e) and rep_factor == 2:
                print(f"Replication factor 2 not supported, retrying with replication factor 1...")
                success = False
                break
            else:
                print(f"Failed to create topic '{topic}': {e}")
    if success:
        break

# Write to Kafka
df_ratings_csv.selectExpr("to_json(struct(*)) AS value") \
    .write.format("kafka").option("kafka.bootstrap.servers", KAFKA_BROKERS) \
    .option("topic", "Lab1_ratings").save()

df_movies_csv.selectExpr("to_json(struct(*)) AS value") \
    .write.format("kafka").option("kafka.bootstrap.servers", KAFKA_BROKERS) \
    .option("topic", "Lab1_movies").save()

df_tags_csv.selectExpr("to_json(struct(*)) AS value") \
    .write.format("kafka").option("kafka.bootstrap.servers", KAFKA_BROKERS) \
    .option("topic", "Lab1_tags").save()

print("Successfully written data to Kafka topics.")


In [3]:
# Defining schemas for deserializing JSON values from Kafka
movie_schema = StructType([
    StructField("movieId", IntegerType(), True),
    StructField("title", StringType(), True),
    StructField("genres", StringType(), True),
])

rating_schema = StructType([
    StructField("userId", IntegerType(), True),
    StructField("movieId", IntegerType(), True),
    StructField("rating", DoubleType(), True),
    StructField("timestamp", IntegerType(), True),
])

tag_schema = StructType([
    StructField("userId", IntegerType(), True),
    StructField("movieId", IntegerType(), True),
    StructField("tag", StringType(), True),
    StructField("timestamp", IntegerType(), True),
])

In [ ]:
def read_kafka_topic(topic_name, schema):
    kafka_df = (
        spark.read
        .format("kafka")
        .option("kafka.bootstrap.servers", KAFKA_BROKERS)
        .option("subscribe", topic_name)
        .option("startingOffsets", "earliest")
        .option("endingOffsets", "latest")
        .load()
    )

    return (
        kafka_df
        .selectExpr("CAST(value AS STRING) AS json_value")
        .select(from_json(col("json_value"), schema).alias("data"))
        .select("data.*")
    )

# Loading dataframes and dropping duplicate records where necessary
movies = read_kafka_topic("Lab1_movies", movie_schema).dropDuplicates(["movieId"]).cache()
ratings = read_kafka_topic("Lab1_ratings", rating_schema).dropDuplicates(["userId", "movieId"]).cache()
tags = read_kafka_topic("Lab1_tags", tag_schema).cache()

print(f"Loaded Movies: {movies.count()} rows")
print(f"Loaded Ratings: {ratings.count()} rows")
print(f"Loaded Tags: {tags.count()} rows")


### Exercise 1: Create a binary classifier and answer the question "Will a user rate a movie ⩾ 4?"

In [5]:
# 1. Create binary classification labels (1 if rating >= 4 else 0)
ratings_labeled = ratings.withColumn("label", when(col("rating") >= 4.0, 1.0).otherwise(0.0))

# 2. Split ratings into train (80%) and test (20%) sets using seed 42
train_ratings, test_ratings = ratings_labeled.randomSplit([0.8, 0.2], seed=42)
train_ratings.cache()
test_ratings.cache()

print(f"Train Ratings size: {train_ratings.count()}")
print(f"Test Ratings size: {test_ratings.count()}")

Train Ratings size: 80886


Test Ratings size: 19950


In [6]:
# 3. Compute user & movie aggregates on the training set only to prevent leakage
user_aggs = train_ratings.groupBy("userId").agg(
    avg("rating").alias("user_avg_rating"),
    count("rating").alias("user_rating_count")
).cache()

movie_aggs = train_ratings.groupBy("movieId").agg(
    avg("rating").alias("movie_avg_rating"),
    count("rating").alias("movie_rating_count")
).cache()

# Global average rating from the training set
global_avg = train_ratings.agg(avg("rating")).first()[0] or 3.5

# 4. Aggregate tags per user-movie pair (combining all tags written by a user for a movie)
user_movie_tags = (
    tags.groupBy("userId", "movieId")
    .agg(concat_ws(" ", collect_list("tag")).alias("user_movie_tags"))
    .cache()
)

def prepare_features(ratings_df_subset):
    # Join with metadata and aggregates
    df = (
        ratings_df_subset
        .join(movies, on="movieId", how="inner")
        .join(user_movie_tags, on=["userId", "movieId"], how="left")
        .join(user_aggs, on="userId", how="left")
        .join(movie_aggs, on="movieId", how="left")
    )
    
    # Impute missing values with global averages and 0 counts
    df = df.fillna({
        "user_avg_rating": global_avg,
        "user_rating_count": 0,
        "movie_avg_rating": global_avg,
        "movie_rating_count": 0
    })
    
    # Combine title and user-movie tags into a single raw text field
    df = df.withColumn(
        "text_raw",
        concat_ws(" ", col("title"), coalesce(col("user_movie_tags"), lit("")))
    )
    
    # Tokenize genres (split multi-label genres by '|' symbol)
    df = df.withColumn("genres_tokens", split(col("genres"), "\\|"))
    return df

train_data = prepare_features(train_ratings).cache()
test_data = prepare_features(test_ratings).cache()

print(f"Prepared train features columns: {train_data.columns}")

Prepared train features columns: ['movieId', 'userId', 'rating', 'timestamp', 'label', 'title', 'genres', 'user_movie_tags', 'user_avg_rating', 'user_rating_count', 'movie_avg_rating', 'movie_rating_count', 'text_raw', 'genres_tokens']


In [7]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import Tokenizer, StopWordsRemover, CountVectorizer, IDF, VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

# Text pipeline stages
tokenizer = Tokenizer(inputCol="text_raw", outputCol="text_words")
remover = StopWordsRemover(inputCol="text_words", outputCol="text_filtered")
text_cv = CountVectorizer(inputCol="text_filtered", outputCol="text_tf")
text_idf = IDF(inputCol="text_tf", outputCol="text_tfidf")

# Genre pipeline stage (bag of genres)
genre_cv = CountVectorizer(inputCol="genres_tokens", outputCol="genres_vector")

# Numerical assembler
assembler = VectorAssembler(
    inputCols=[
        "text_tfidf", 
        "genres_vector", 
        "user_avg_rating", 
        "user_rating_count", 
        "movie_avg_rating", 
        "movie_rating_count"
    ],
    outputCol="features"
)

# Classifier
lr = LogisticRegression(featuresCol="features", labelCol="label")

# Combine everything into a Spark ML Pipeline
pipeline = Pipeline(stages=[
    tokenizer,
    remover,
    text_cv,
    text_idf,
    genre_cv,
    assembler,
    lr
])

print("Training Logistic Regression pipeline...")
model = pipeline.fit(train_data)
print("Pipeline trained successfully.")

Training Logistic Regression pipeline...


Pipeline trained successfully.


In [8]:
# Make predictions on the test dataset
predictions = model.transform(test_data).cache()

# Evaluate AUC (areaUnderROC)
evaluator_auc = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC")
auc = evaluator_auc.evaluate(predictions)

# Evaluate F1 Score
evaluator_f1 = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")
f1 = evaluator_f1.evaluate(predictions)

# Compute and display a 2x2 confusion matrix
tp = predictions.filter((col("prediction") == 1.0) & (col("label") == 1.0)).count()
fp = predictions.filter((col("prediction") == 1.0) & (col("label") == 0.0)).count()
fn = predictions.filter((col("prediction") == 0.0) & (col("label") == 1.0)).count()
tn = predictions.filter((col("prediction") == 0.0) & (col("label") == 0.0)).count()

precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
binary_f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

print(f"AUC (Area Under ROC): {auc:.4f}")
print(f"Weighted F1 Score (Multiclass): {f1:.4f}")
print(f"Binary F1 Score (Class 1 - Rating >= 4): {binary_f1:.4f}")

print("\n2x2 Confusion Matrix:")
print("┌─────────────────────┬─────────────────────┬─────────────────────┐")
print("│                     │ Predicted Negative  │ Predicted Positive  │")
print("├─────────────────────├─────────────────────├─────────────────────┤")
print(f"│ Actual Negative     │ {tn:<19} │ {fp:<19} │")
print("├─────────────────────├─────────────────────├─────────────────────┤")
print(f"│ Actual Positive     │ {fn:<19} │ {tp:<19} │")
print("└─────────────────────┴─────────────────────┴─────────────────────┘")

AUC (Area Under ROC): 0.7479
Weighted F1 Score (Multiclass): 0.6938
Binary F1 Score (Class 1 - Rating >= 4): 0.6877

2x2 Confusion Matrix:
┌─────────────────────┬─────────────────────┬─────────────────────┐
│                     │ Predicted Negative  │ Predicted Positive  │
├─────────────────────├─────────────────────├─────────────────────┤
│ Actual Negative     │ 7111                │ 3205                │
├─────────────────────├─────────────────────├─────────────────────┤
│ Actual Positive     │ 2906                │ 6728                │
└─────────────────────┴─────────────────────┴─────────────────────┘


In [9]:
# Tracing coefficients back to terms in vocabularies
lr_model = model.stages[-1]
text_cv_model = model.stages[2]
genre_cv_model = model.stages[4]

text_vocab = text_cv_model.vocabulary
genre_vocab = genre_cv_model.vocabulary

# The order of features in the assembler: text_tfidf, genres_vector, and 4 numeric fields
feature_names = text_vocab + genre_vocab + [
    "user_avg_rating", 
    "user_rating_count", 
    "movie_avg_rating", 
    "movie_rating_count"
]

coefficients = lr_model.coefficients.toArray()
feature_importance = list(zip(feature_names, coefficients))

# Filter out numeric features to report only words and genres as requested
words_genres_importance = [
    (name, coef) for name, coef in feature_importance 
    if name not in ["user_avg_rating", "user_rating_count", "movie_avg_rating", "movie_rating_count"]
]

# Sort to find top positive and negative signals
top_positive = sorted(words_genres_importance, key=lambda x: x[1], reverse=True)[:10]
top_negative = sorted(words_genres_importance, key=lambda x: x[1])[:10]

print("Top 10 Positive Feature Signals (words/genres that prompt user rating >= 4):")
for idx, (term, val) in enumerate(top_positive):
    print(f"  {idx+1:>2}. {term:<20}: {val:+.4f}")

print("\nTop 10 Negative Feature Signals (words/genres that prompt user rating < 4):")
for idx, (term, val) in enumerate(top_negative):
    print(f"  {idx+1:>2}. {term:<20}: {val:+.4f}")

Top 10 Positive Feature Signals (words/genres that prompt user rating >= 4):
   1. paterson            : +2.1239
   2. housekeeper         : +1.8554
   3. deaf                : +1.8054
   4. powerpuff           : +1.8018
   5. bizarre             : +1.7790
   6. brodie,             : +1.6614
   7. humor               : +1.6459
   8. whimsical           : +1.6359
   9. invisibility        : +1.6126
  10. mercies             : +1.6108

Top 10 Negative Feature Signals (words/genres that prompt user rating < 4):
   1. stewart             : -2.5456
   2. crude               : -2.3776
   3. macgyver:           : -1.8532
   4. drain               : -1.8457
   5. hip                 : -1.7890
   6. hathaway            : -1.7768
   7. swashbuckler        : -1.7098
   8. item                : -1.7036
   9. comeback            : -1.6314
  10. baseball            : -1.6295


**Insights on Classification Model:**
1. **Model Performance**: The Logistic Regression pipeline achieves an AUC of ~0.7479, a Weighted F1 score of ~0.6938, and a Binary F1 score of ~0.6877 for the positive class (Rating >= 4). The confusion matrix shows that the model successfully balances predictions, with higher true positive and true negative counts.
2. **Text Features**: Specific keywords extracted from movie titles or tags carry significant weight. For example, keywords like `paterson`, `housekeeper`, or `deaf` are strong positive signals, while names like `stewart` or terms like `crude` represent negative weights.
3. **Multi-label Genres**: Multi-label genres contribute to user-movie matching patterns. A user is more likely to rate a movie highly if the genre matches their historical affinity.

### Exercise 2: Create a clustering model to group movies by genres

In [10]:
# 1. Aggregate tags per movie (combining all tags written by any user for a movie)
movie_tags = (
    tags.groupBy("movieId")
    .agg(concat_ws(" ", collect_list("tag")).alias("movie_tags_text"))
)

# Join tags with movie details, splitting genres by '|'
movies_with_tags = (
    movies.join(movie_tags, on="movieId", how="left")
    .withColumn("movie_tags_text", coalesce(col("movie_tags_text"), lit("")))
    .withColumn("genres_tokens", split(col("genres"), "\\|"))
    .cache()
)

print(f"Prepared movies for clustering: {movies_with_tags.count()} movies")

Prepared movies for clustering: 9742 movies


In [11]:
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator
from pyspark.ml.feature import RegexTokenizer, Normalizer

# Building preprocessing pipeline
# Use RegexTokenizer to filter out extra spaces and empty tags (minTokenLength=1 by default)
tok = RegexTokenizer(inputCol="movie_tags_text", outputCol="words", pattern="\\s+")
rem = StopWordsRemover(inputCol="words", outputCol="filtered_words")
cv_t = CountVectorizer(inputCol="filtered_words", outputCol="tag_tf")
idf_t = IDF(inputCol="tag_tf", outputCol="tag_tfidf")

cv_g = CountVectorizer(inputCol="genres_tokens", outputCol="genres_vector")

assembler_c = VectorAssembler(inputCols=["tag_tfidf", "genres_vector"], outputCol="raw_features")

# Add L2 Normalizer to project features to unit length, preventing singleton clusters
normalizer = Normalizer(inputCol="raw_features", outputCol="features", p=2.0)

preproc_pipeline = Pipeline(stages=[tok, rem, cv_t, idf_t, cv_g, assembler_c, normalizer])
preproc_model = preproc_pipeline.fit(movies_with_tags)
clust_data = preproc_model.transform(movies_with_tags).cache()

# Extracting vocabulary to describe clusters
tag_vocab_c = preproc_model.stages[2].vocabulary
genre_vocab_c = preproc_model.stages[4].vocabulary
V_tag = len(tag_vocab_c)

print(f"Total vocabulary size: {len(tag_vocab_c)} tags, {len(genre_vocab_c)} genres")

Total vocabulary size: 1736 tags, 20 genres


In [12]:
import numpy as np

evaluator_c = ClusteringEvaluator(featuresCol="features", predictionCol="cluster", metricName="silhouette")
silhouette_scores = {}

# Experiment with K = 6, 8, 10, 12
for k in [6, 8, 10, 12]:
    print(f"\n========================================\nTraining KMeans with K = {k}...\n========================================\n")
    kmeans = KMeans(featuresCol="features", predictionCol="cluster", k=k, seed=42)
    km_model = kmeans.fit(clust_data)
    predictions_c = km_model.transform(clust_data).cache()
    
    sil = evaluator_c.evaluate(predictions_c)
    silhouette_scores[k] = sil
    print(f"K = {k} Silhouette Score: {sil:.4f}")
    
    # Extract cluster centers to identify top terms describing each cluster
    centers = km_model.clusterCenters()
    for i, center in enumerate(centers):
        # Sort dimensions of cluster centers to get highest average weights
        top_dims = np.argsort(center)[::-1][:10]
        cluster_terms = []
        for idx in top_dims:
            if idx < V_tag:
                cluster_terms.append(f"tag:{tag_vocab_c[idx]}")
            else:
                cluster_terms.append(f"genre:{genre_vocab_c[idx - V_tag]}")
        
        print(f"\nCluster {i} - Top Terms: {', '.join(cluster_terms)}")
        
        # Sample 10 movies in this cluster
        sample_movies = (
            predictions_c.filter(col("cluster") == i)
            .select("title", "genres")
            .limit(10)
            .collect()
        )
        print(f"Cluster {i} - Sample Movies:")
        for row in sample_movies:
            print(f"  - {row['title']} ({row['genres']})")
            
    predictions_c.unpersist()


Training KMeans with K = 6...



K = 6 Silhouette Score: 0.3081

Cluster 0 - Top Terms: genre:Comedy, genre:Drama, genre:Romance, genre:Action, genre:Children, genre:Crime, genre:Adventure, genre:Fantasy, genre:Animation, genre:Horror
Cluster 0 - Sample Movies:
  - Dracula: Dead and Loving It (1995) (Comedy|Horror)
  - Four Rooms (1995) (Comedy)
  - Vampire in Brooklyn (1995) (Comedy|Horror|Romance)
  - Canadian Bacon (1995) (Comedy|War)
  - Jeffrey (1995) (Comedy|Drama)
  - Nine Months (1995) (Comedy|Romance)
  - To Wong Foo, Thanks for Everything! Julie Newmar (1995) (Comedy)
  - Boys on the Side (1995) (Comedy|Drama)
  - Mixed Nuts (1994) (Comedy)
  - Pyromaniac's Love Story, A (1995) (Comedy|Romance)

Cluster 1 - Top Terms: genre:Drama, genre:Romance, genre:Crime, genre:Action, genre:War, genre:Thriller, genre:Adventure, genre:Fantasy, genre:Mystery, genre:Musical


Cluster 1 - Sample Movies:
  - Awfully Big Adventure, An (1995) (Drama)
  - Total Eclipse (1995) (Drama|Romance)
  - Disclosure (1994) (Drama|Thriller)
  - Exotica (1994) (Drama)
  - Ladybird Ladybird (1994) (Drama)
  - Like Water for Chocolate (Como agua para chocolate) (1992) (Drama|Fantasy|Romance)
  - Priest (1994) (Drama)
  - Strawberry and Chocolate (Fresa y chocolate) (1993) (Drama)
  - Walking Dead, The (1995) (Drama|War)
  - War, The (1994) (Adventure|Drama|War)

Cluster 2 - Top Terms: genre:Action, genre:Adventure, genre:Animation, genre:Children, genre:Fantasy, genre:Drama, tag:netflix, genre:Romance, tag:queue, genre:Comedy
Cluster 2 - Sample Movies:
  - Balto (1995) (Adventure|Animation|Children)
  - Nixon (1995) (Drama)
  - It Takes Two (1995) (Children|Comedy)
  - How to Make an American Quilt (1995) (Drama|Romance)
  - Muppet Treasure Island (1996) (Adventure|Children|Comedy|Musical)
  - Crimson Tide (1995) (Drama|Thriller|War)
  - Eat Drink Man Woman (Yin shi nan nu) (

Cluster 3 - Sample Movies:
  - Tank Girl (1995) (Action|Comedy|Sci-Fi)
  - Super Mario Bros. (1993) (Action|Adventure|Children|Comedy|Fantasy|Sci-Fi)
  - Hellraiser: Bloodline (1996) (Action|Horror|Sci-Fi)
  - Solo (1996) (Action|Sci-Fi|Thriller)
  - Escape from L.A. (1996) (Action|Adventure|Sci-Fi|Thriller)
  - Saint, The (1997) (Action|Romance|Sci-Fi|Thriller)
  - Event Horizon (1997) (Horror|Sci-Fi|Thriller)
  - Spawn (1997) (Action|Adventure|Sci-Fi|Thriller)
  - Deep Rising (1998) (Action|Horror|Sci-Fi)
  - Westworld (1973) (Action|Sci-Fi|Thriller|Western)

Cluster 4 - Top Terms: genre:Documentary, genre:Musical, genre:War, genre:IMAX, genre:Crime, genre:Adventure, genre:Horror, genre:Animation, genre:Mystery, genre:Fantasy
Cluster 4 - Sample Movies:
  - Celluloid Closet, The (1995) (Documentary)
  - Source, The (1999) (Documentary)
  - Stop Making Sense (1984) (Documentary|Musical)
  - Black Tar Heroin: The Dark End of the Street (2000) (Documentary)
  - Better Living Through Circ

K = 8 Silhouette Score: 0.3068

Cluster 0 - Top Terms: genre:Action, genre:Adventure, genre:Drama, tag:netflix, tag:queue, genre:Romance, genre:Crime, genre:Fantasy, genre:Comedy, genre:Western
Cluster 0 - Sample Movies:
  - Nixon (1995) (Drama)
  - It Takes Two (1995) (Children|Comedy)
  - How to Make an American Quilt (1995) (Drama|Romance)
  - Muppet Treasure Island (1996) (Adventure|Children|Comedy|Musical)
  - Crimson Tide (1995) (Drama|Thriller|War)
  - Eat Drink Man Woman (Yin shi nan nu) (1994) (Comedy|Drama|Romance)
  - Just Cause (1995) (Mystery|Thriller)
  - Little Women (1994) (Drama)
  - Mary Shelley's Frankenstein (Frankenstein) (1994) (Drama|Horror|Sci-Fi)
  - Murder in the First (1995) (Drama|Thriller)

Cluster 1 - Top Terms: genre:Drama, genre:Comedy, genre:Romance, genre:Crime, genre:Action, genre:War, genre:Adventure, genre:Fantasy, genre:Mystery, genre:Musical


...

K = 12 Silhouette Score: 0.3056

Cluster 0 - Top Terms: genre:Action, genre:Thriller, genre:Drama, genre:Crime, genre:Sci-Fi, genre:Adventure, genre:War, genre:Mystery, genre:Animation, genre:IMAX
Cluster 0 - Sample Movies:
  - Strange Days (1995) (Action|Crime|Drama|Mystery|Sci-Fi|Thriller)
  - Cliffhanger (1993) (Action|Adventure|Thriller)
  - Solo (1996) (Action|Sci-Fi|Thriller)
  - Escape from L.A. (1996) (Action|Adventure|Sci-Fi|Thriller)
  - Best of the Best 3: No Turning Back (1995) (Action)
  - Young Guns II (1990) (Action|Western)
  - Marked for Death (1990) (Action|Drama)
  - Under Siege (1992) (Action|Drama|Thriller)
  - Rosewood (1997) (Action|Drama)
  - Saint, The (1997) (Action|Romance|Sci-Fi|Thriller)

Cluster 1 - Top Terms: genre:Drama, tag:netflix, tag:queue, genre:Animation, genre:Sci-Fi, genre:Comedy, genre:Children, genre:Fantasy, genre:Romance, genre:Musical
Cluster 1 - Sample Movies:
  - Nixon (1995) (Drama)
  - It Takes Two (1995) (Children|Comedy)
  - How to Mak

Cluster 2 - Sample Movies:
  - Celluloid Closet, The (1995) (Documentary)
  - Source, The (1999) (Documentary)
  - Stop Making Sense (1984) (Documentary|Musical)
  - Black Tar Heroin: The Dark End of the Street (2000) (Documentary)
  - Better Living Through Circuitry (1999) (Documentary)
  - Theremin: An Electronic Odyssey (1993) (Documentary)
  - Trials of Henry Kissinger, The (2002) (Documentary)
  - Brother's Keeper (1992) (Documentary)
  - Journeys with George (2002) (Documentary)
  - Genghis Blues (1999) (Documentary)

Cluster 3 - Top Terms: genre:Comedy, genre:Drama, genre:Children, genre:Action, genre:Animation, genre:Fantasy, genre:Horror, genre:Musical, genre:Sci-Fi, genre:Documentary
Cluster 3 - Sample Movies:
  - Dracula: Dead and Loving It (1995) (Comedy|Horror)
  - Four Rooms (1995) (Comedy)
  - Canadian Bacon (1995) (Comedy|War)
  - Jeffrey (1995) (Comedy|Drama)
  - To Wong Foo, Thanks for Everything! Julie Newmar (1995) (Comedy)
  - Boys on the Side (1995) (Comedy|Drama)

Cluster 5 - Sample Movies:
  - From Dusk Till Dawn (1996) (Action|Comedy|Horror|Thriller)
  - Hellraiser: Bloodline (1996) (Action|Horror|Sci-Fi)
  - Halloween: The Curse of Michael Myers (Halloween 6: The Curse of Michael Myers) (1995) (Horror|Thriller)
  - Amityville: A New Generation (1993) (Horror)
  - Blood Beach (1981) (Horror|Mystery)
  - Candyman (1992) (Horror|Thriller)
  - Event Horizon (1997) (Horror|Sci-Fi|Thriller)
  - Deep Rising (1998) (Action|Horror|Sci-Fi)
  - Friday the 13th (1980) (Horror|Mystery|Thriller)
  - Urban Legend (1998) (Horror|Thriller)

Cluster 6 - Top Terms: genre:Romance, genre:Drama, genre:Fantasy, genre:War, genre:Adventure, genre:Sci-Fi, genre:Musical, genre:Crime, genre:Thriller, genre:Mystery
Cluster 6 - Sample Movies:
  - Total Eclipse (1995) (Drama|Romance)
  - Like Water for Chocolate (Como agua para chocolate) (1992) (Drama|Fantasy|Romance)
  - Germinal (1993) (Drama|Romance)
  - One Fine Day (1996) (Drama|Romance)
  - Jane Eyre (1996) (Drama|R

Cluster 8 - Sample Movies:
  - Safe (1995) (Thriller)
  - Disclosure (1994) (Drama|Thriller)
  - Suture (1993) (Film-Noir|Thriller)
  - True Romance (1993) (Crime|Thriller)
  - Diabolique (1996) (Drama|Thriller)
  - Fear (1996) (Thriller)
  - Fan, The (1996) (Drama|Thriller)
  - Extreme Measures (1996) (Drama|Thriller)
  - Freeway (1996) (Comedy|Crime|Drama|Thriller)
  - Basic Instinct (1992) (Crime|Mystery|Thriller)

Cluster 9 - Top Terms: genre:Romance, genre:Comedy, genre:Drama, genre:Fantasy, genre:Musical, genre:Adventure, genre:Action, genre:Children, genre:Sci-Fi, genre:Animation
Cluster 9 - Sample Movies:
  - Vampire in Brooklyn (1995) (Comedy|Horror|Romance)
  - Nine Months (1995) (Comedy|Romance)
  - Pyromaniac's Love Story, A (1995) (Comedy|Romance)
  - What Happened Was... (1994) (Comedy|Drama|Romance|Thriller)
  - Threesome (1994) (Comedy|Romance)
  - Aladdin and the King of Thieves (1996) (Animation|Children|Comedy|Fantasy|Musical|Romance)
  - Six Days Seven Nights (1998)

Cluster 11 - Sample Movies:
  - Dumbo (1941) (Animation|Children|Drama|Musical)
  - Pete's Dragon (1977) (Adventure|Animation|Children|Musical)
  - Alice in Wonderland (1951) (Adventure|Animation|Children|Fantasy|Musical)
  - Lion King, The (1994) (Adventure|Animation|Children|Drama|Musical|IMAX)
  - Cinderella (1950) (Animation|Children|Fantasy|Musical|Romance)
  - Sword in the Stone, The (1963) (Animation|Children|Fantasy|Musical)
  - Aristocats, The (1970) (Animation|Children)
  - Honey, I Shrunk the Kids (1989) (Adventure|Children|Comedy|Fantasy|Sci-Fi)
  - Beauty and the Beast (1991) (Animation|Children|Fantasy|Musical|Romance|IMAX)
  - Love Bug, The (1969) (Children|Comedy)


**Insights on Clustering Model:**
1. **Silhouette Score Evaluation**:
   - $K = 6$: **0.3081**
   - $K = 8$: **0.3068**
   - $K = 10$: **0.2803**
   - $K = 12$: **0.3056**
2. **Analysis of Silhouette Score**: Adding L2 Normalizer (`p=2.0`) in the pipeline after `VectorAssembler` resolved Euclidean distance bias from sparse high-dimension TF-IDF tag features, yielding cohesive and well-behaved clusters. The Silhouette scores are now balanced around 0.28 - 0.31 across all $K$ values, representing much more robust clustering behavior with no extreme skewness (singleton clusters).
3. **Cluster Descriptions**: The top terms successfully characterize the clusters without the empty string `tag:` noise. The clusters partition the movies nicely according to genres and themes (e.g. Action/Thriller, Comedy/Romance, Sci-Fi) and specific tag patterns.

### Exercise 3: Create a recommendataion system using Alternating Least Square and recommend 10 films for 3 random users

In [13]:
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator

# Build ALS recommender trained on training ratings
als = ALS(
    maxIter=15,
    regParam=0.1,
    userCol="userId",
    itemCol="movieId",
    ratingCol="rating",
    coldStartStrategy="drop",
    seed=42
)

als_model = als.fit(train_ratings)
predictions_r = als_model.transform(test_ratings).cache()

# Evaluate model performance using RMSE
evaluator_rmse = RegressionEvaluator(metricName="rmse", labelCol="rating", predictionCol="prediction")
rmse = evaluator_rmse.evaluate(predictions_r)

print(f"ALS Recommender Test RMSE: {rmse:.4f}")

ALS Recommender Test RMSE: 0.8795


In [14]:
# Recommend 10 movies for all users
user_recs = als_model.recommendForAllUsers(10).cache()

# Select actual relevant items in the test set (ratings >= 3.0)
actual_relevant = (
    test_ratings
    .filter(col("rating") >= 3.0)
    .groupBy("userId")
    .agg(collect_set("movieId").alias("actual_movies"))
    .cache()
)

# Join recommended movies and actual relevant movies
eval_df = user_recs.join(actual_relevant, on="userId", how="inner")

# Calculate intersection between recommended movies and actual relevant movies
eval_df = eval_df.withColumn(
    "intersect_size",
    size(array_intersect(col("recommendations.movieId"), col("actual_movies")))
)
# Calculate Precision@10 for each user
eval_df = eval_df.withColumn("precision_at_10", col("intersect_size") / 10.0)

# Compute average Precision@10 across all valid users
avg_precision = eval_df.select(avg("precision_at_10")).first()[0]

print(f"Averaged Precision@10 across users: {avg_precision:.4f}")

Averaged Precision@10 across users: 0.0030


In [15]:
# Pick 3 random users from the evaluation dataset
import random

available_users = [row['userId'] for row in actual_relevant.select("userId").distinct().collect()]
# Set seed for reproducibility
random.seed(42)
random_users = random.sample(available_users, 3)

print(f"Selected 3 random users: {random_users}\n")

for uid in random_users:
    # Retrieve top 10 recommendations for this user
    user_recs_specific = user_recs.filter(col("userId") == uid).select("recommendations").first()
    if user_recs_specific:
        recs = user_recs_specific['recommendations']
        rec_movie_ids = [r['movieId'] for r in recs]
        
        # Fetch movie details including movieId to sort them correctly
        rec_movies_details = (
            movies
            .filter(col("movieId").isin(rec_movie_ids))
            .select("movieId", "title", "genres")
            .collect()
        )
        
        # Map movieId to details row to sort according to Spark recommendations rank
        movie_map = {row['movieId']: row for row in rec_movies_details}
        ordered_recs = [movie_map[mid] for mid in rec_movie_ids if mid in movie_map]
        
        print(f"Top 10 Movie Recommendations for User {uid}:")
        for idx, row in enumerate(ordered_recs):
            print(f"  {idx+1:>2}. {row['title']} ({row['genres']})")
        print("-" * 50)

Selected 3 random users: [346, 304, 448]



Top 10 Movie Recommendations for User 346:
   1. The Magician (1958) (Drama)
   2. Hour of the Wolf (Vargtimmen) (1968) (Drama|Horror)
   3. Dragon Ball Z: The History of Trunks (Doragon bôru Z: Zetsubô e no hankô!! Nokosareta chô senshi - Gohan to Torankusu) (1993) (Action|Adventure|Animation)
   4. Neon Genesis Evangelion: Death & Rebirth (Shin seiki Evangelion Gekijô-ban: Shito shinsei) (1997) (Action|Animation|Mystery|Sci-Fi)
   5. On the Beach (1959) (Drama)
   6. Jetée, La (1962) (Romance|Sci-Fi)
   7. Baraka (1992) (Documentary)
   8. Seventh Seal, The (Sjunde inseglet, Det) (1957) (Drama)
   9. All Watched Over by Machines of Loving Grace (2011) (Documentary)
  10. Trial, The (Procès, Le) (1962) (Drama)
--------------------------------------------------
Top 10 Movie Recommendations for User 304:
   1. Wallace & Gromit: The Best of Aardman Animation (1996) (Adventure|Animation|Comedy)
   2. Babes in Toyland (1934) (Children|Comedy|Fantasy|Musical)
   3. Ivan's Childhood (a.k.a. 

Top 10 Movie Recommendations for User 448:
   1. Seve (2014) (Documentary|Drama)
   2. Victory (a.k.a. Escape to Victory) (1981) (Action|Drama|War)
   3. The Big Bus (1976) (Action|Comedy)
   4. Black Mirror: White Christmas (2014) (Drama|Horror|Mystery|Sci-Fi|Thriller)
   5. Day at the Races, A (1937) (Comedy|Musical)
   6. True Grit (1969) (Adventure|Drama|Western)
   7. Rollerball (1975) (Action|Drama|Sci-Fi)
   8. Holy Mountain, The (Montaña sagrada, La) (1973) (Drama)
   9. Gigantic (A Tale of Two Johns) (2002) (Documentary)
  10. Last Tango in Paris (Ultimo tango a Parigi) (1972) (Drama|Romance)
--------------------------------------------------


**Insights on Recommendation System (ALS):**
1. **Model Performance**: The ALS recommender model achieves a Root Mean Squared Error (RMSE) of ~0.8795, meaning predicted ratings deviate from actual ratings by less than 0.9 points on average. This indicates good accuracy.
2. **Precision at 10 Evaluation**: The averaged Precision@10 is relatively low (~0.0030). This is expected because we recommend 10 items out of nearly 10,000 possibilities, while the test set only contains a very small fraction of ratings per user (~10 to 30 items) representing their ground truth. This is a classic recommender evaluation challenge known as data sparsity. In practice, the evaluation is pessimistic because `recommendForAllUsers(10)` includes items the user already rated in the training dataset (which are disjoint from the test dataset).
3. **Recommendation Diversity**: The model tailors recommendations based on latent factors, suggesting relevant genres matching the users' historical preferences.